In [ ]:
# 1 安装基础的依赖包
sudo yum install -y bcc bpftrace ipmitool smartmontools lm_sensors \
                    stress-ng chaosblade docker-ce prometheus2 grafana \
                    python3.10 python3.10-pip git

In [ ]:
# 2. 全栈观测体系搭建
搭建H/K/R/A 四层统一观测平台，采集 BCPN 模型所需的所有指标，采样间隔严格设置为 10 秒（与论文一致）。
## 2.1 部署 Prometheus+Grafana
# 启动Prometheus
sudo systemctl enable --now prometheus
sudo systemctl enable --now grafana-server

# 配置Prometheus采集目标（/etc/prometheus/prometheus.yml）
sudo tee /etc/prometheus/prometheus.yml <<EOF
global:
  scrape_interval: 10s  # 严格对齐论文采样间隔
  evaluation_interval: 10s

scrape_configs:
  - job_name: 'node'  # 硬件层+内核层指标
    static_configs:
      - targets: ['localhost:9100']
  - job_name: 'cadvisor'  # 运行时层（容器）指标
    static_configs:
      - targets: ['localhost:8080']
  - job_name: 'app'  # 应用层指标
    static_configs:
      - targets: ['localhost:8000']
EOF

sudo systemctl restart prometheus

# 检查服务状态
systemctl status prometheus --no-pager
# 预期：active (running)

# 检查端口监听
ss -tulpn | grep 9090
# 预期：LISTEN 0.0.0.0:9090

# 访问API验证
curl -s http://localhost:9090/api/v1/status/config | head -20
# 预期：返回JSON格式的配置信息

In [ ]:
## 2.2 部署各层指标采集器
### （1）硬件层 + 内核层：Node Exporter,采集指标：CPU 使用率、内存使用率、磁盘 IO、磁盘温度、网络流量、内核调度延迟等。
docker run -d --name node-exporter --net=host --pid=host \
  -v "/:/host:ro,rslave" \
  quay.io/prometheus/node-exporter:latest \
  --path.rootfs=/host


# 检查容器状态
docker ps | grep node-exporter
# 预期：运行中，端口9100

# 检查指标输出
curl -s http://localhost:9100/metrics | grep node_disk_io_time_seconds_total
# 预期：返回类似 node_disk_io_time_seconds_total{device="sda"} 12345.67 的数值

In [ ]:
### （2）运行时层：cAdvisor, 采集指标：容器 CPU / 内存 / IO 使用率、容器重启次数、OOM 事件等。
docker run -d --name cadvisor --net=host \
  -v /:/rootfs:ro \
  -v /var/run:/var/run:ro \
  -v /sys:/sys:ro \
  -v /var/lib/docker/:/var/lib/docker:ro \
  gcr.io/cadvisor/cadvisor:latest

# 查看容器状态
docker ps | grep cadvisor

# 访问 cAdvisor Web 界面（默认端口 8080）
curl http://localhost:8080/metrics

In [ ]:
# 跑起数据库应用
# 启动 MySQL
docker run -d --name mysql \
  -e MYSQL_ROOT_PASSWORD=123456 \
  -e MYSQL_DATABASE=order_db \
  -p 3306:3306 \
  mysql:8.0

# 等待 MySQL 启动（约10秒）
sleep 10

# 初始化订单表（进入 MySQL 容器执行）
docker exec -i mysql mysql -uroot -p123456 order_db <<EOF
CREATE TABLE IF NOT EXISTS orders (
    id INT AUTO_INCREMENT PRIMARY KEY,
    user_id INT NOT NULL,
    product_id INT NOT NULL,
    amount DECIMAL(10,2) NOT NULL,
    status VARCHAR(20) NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    INDEX idx_user (user_id),
    INDEX idx_product (product_id)
) ENGINE=InnoDB;
EOF

In [ ]:
# 应用加压
python app_exporter.py & 

# 数据库加压脚本：
python app_exporter_with_mysql_heavy.py

# 在上两个操作的3分钟后，执行
# 3 故障 1：硬件层 SSD 介质坏道（论文核心案例）
# 方法1：用dd模拟磁盘IO饱和（安全）
sudo dd if=/dev/sda of=/dev/null bs=1M count=100000 &

# 方法2：用chaosblade注入磁盘IO错误（更真实）
curl -L https://chaosblade.oss-cn-hangzhou.aliyuncs.com/agent/github/1.7.2/chaosblade-1.7.2-linux-amd64.tar.gz | tar xz
cd chaosblade-1.7.2-linux-amd64
./blade create disk fill --path=/ --size=1024 --timeout=300s

# 执行上述操作后，等待5-10分钟，然后执行生成数据脚本
python generate_bcpn_data.py


In [ ]:
# app_exporter_with_mysql_heavy.py
from prometheus_client import start_http_server, Counter, Gauge
import time
import random
import pymysql
from threading import Thread, Lock

# 应用层指标
REQUESTS = Counter('app_requests_total', 'Total requests')
RETRIES = Counter('app_retries_total', 'Total retries')
RESPONSE_TIME = Gauge('app_response_time_ms', 'Average response time')
DB_ERRORS = Counter('app_db_errors_total', 'Database errors')

# MySQL 连接配置
DB_CONFIG = {
    'host': '127.0.0.1',
    'port': 3306,
    'user': 'root',
    'password': '123456',
    'database': 'order_db',
    'charset': 'utf8mb4',
    'cursorclass': pymysql.cursors.DictCursor,
    'autocommit': False  # 关闭自动提交，增加事务开销
}

# 全局锁（用于线程安全）
lock = Lock()
# 预生成大量测试数据（内存中）
TEST_DATA = [(random.randint(1, 100000), 
              random.randint(1, 10000), 
              round(random.uniform(10, 10000), 2), 
              random.choice(['pending', 'paid', 'shipped', 'cancelled', 'refunded'])) 
             for _ in range(10000)]

def init_heavy_data():
    """【初始化】先插入100万条数据，让表变大，查询变慢"""
    print("正在初始化100万条测试数据（约需2分钟）...")
    conn = pymysql.connect(**DB_CONFIG)
    try:
        with conn.cursor() as cursor:
            # 批量插入100万条
            for i in range(100):
                batch = TEST_DATA * 100  # 每次1万条
                cursor.executemany(
                    "INSERT INTO orders (user_id, product_id, amount, status) VALUES (%s, %s, %s, %s)",
                    batch
                )
                conn.commit()
                print(f"已插入 { (i+1)*10000 } 条数据")
    finally:
        conn.close()
    print("✅ 测试数据初始化完成！")

def heavy_db_task():
    """【重负载任务】复杂查询 + 批量更新 + 大事务"""
    conn = None
    try:
        conn = pymysql.connect(**DB_CONFIG)
        
        # 1. 复杂聚合查询（多表关联+排序+分组，非常消耗CPU）
        with conn.cursor() as cursor:
            sql = """
                SELECT 
                    user_id, 
                    COUNT(*) as order_count, 
                    SUM(amount) as total_amount,
                    AVG(amount) as avg_amount
                FROM orders 
                WHERE user_id BETWEEN %s AND %s
                GROUP BY user_id
                HAVING order_count > 5
                ORDER BY total_amount DESC
                LIMIT 100
            """
            cursor.execute(sql, (random.randint(1, 50000), random.randint(50001, 100000)))
            cursor.fetchall()
        
        # 2. 批量更新（大事务，消耗IO+CPU）
        with conn.cursor() as cursor:
            update_ids = [random.randint(1, 1000000) for _ in range(1000)]
            sql = "UPDATE orders SET status = 'updated', amount = amount * 1.01 WHERE id IN (%s)" % ','.join(['%s']*1000)
            cursor.execute(sql, update_ids)
        conn.commit()
        
        # 3. 随机读取（全表扫描概率）
        with conn.cursor() as cursor:
            cursor.execute("SELECT * FROM orders ORDER BY RAND() LIMIT 10")
            cursor.fetchall()
            
    except Exception as e:
        with lock:
            DB_ERRORS.inc()
    finally:
        if conn:
            conn.close()

def worker_thread(thread_id):
    """工作线程：无限循环执行重负载任务"""
    print(f"线程 {thread_id} 启动")
    while True:
        start_time = time.time()
        
        # 执行重负载任务
        heavy_db_task()
        
        # 更新指标
        with lock:
            REQUESTS.inc()
            resp_time = (time.time() - start_time) * 1000
            RESPONSE_TIME.set(max(100, resp_time))
        
        # 【关键】几乎不 sleep，持续施压
        time.sleep(0.001)

if __name__ == '__main__':
    start_http_server(8000)
    fault_start_time = time.time() + 300  # 5分钟后注入故障
    
    # 1. 先初始化100万条数据（仅第一次运行需要）
    # init_heavy_data()  # 注释掉这行如果已经初始化过
    
    # 2. 启动 50 个并发线程（大幅提升并发）
    print("启动 50 个工作线程...")
    for i in range(50):
        t = Thread(target=worker_thread, args=(i,), daemon=True)
        t.start()
    
    # 3. 主线程保持运行
    print("✅ 重负载应用启动成功，指标端口：8000")
    while True:
        time.sleep(1)

In [ ]:
### （3）应用层：自定义 Exporter
编写一个简单的 Python 应用，模拟电商订单服务并暴露指标：
# app_exporter.py
from prometheus_client import start_http_server, Counter, Gauge
import time
import random

# 应用层指标
REQUESTS = Counter('app_requests_total', 'Total requests')
RETRIES = Counter('app_retries_total', 'Total retries')
RESPONSE_TIME = Gauge('app_response_time_ms', 'Average response time')

def mock_order_service():
    """模拟订单服务，故障时会触发重试风暴"""
    REQUESTS.inc()
    # 正常响应时间100ms，故障时500ms以上
    if time.time() > fault_start_time:
        RESPONSE_TIME.set(random.normalvariate(600, 100))
        # 30%概率触发重试
        if random.random() < 0.3:
            RETRIES.inc()
            time.sleep(0.1)
    else:
        RESPONSE_TIME.set(random.normalvariate(100, 20))

if __name__ == '__main__':
    start_http_server(8000)
    fault_start_time = time.time() + 300  # 5分钟后注入故障
    print("应用启动成功，指标端口：8000")
    while True:
        mock_order_service()
        time.sleep(0.1)

In [ ]:
# generate_bcpn_data.py
from prometheus_api_client import PrometheusConnect
import pandas as pd
import json
from datetime import datetime, timedelta
import os  # 补充遗漏的os模块导入

# 配置Prometheus地址
PROM_URL = "http://localhost:9090"
# 故障时间窗口（根据实际故障注入时间调整）
START_TIME = datetime.now() - timedelta(minutes=10)
END_TIME = datetime.now()
STEP = "10s"  # 与采样间隔一致

# 定义BCPN四层节点与对应Prometheus指标
BCPN_NODES = [
    {
        "node_id": "host01_ssd_io_util",
        "layer": "H",
        "metric": "node_disk_io_time_seconds_total{device=\"sda\"}",
        "has_inherent_defect": True,
        "node_description": "SSD磁盘IO利用率"
    },
    {
        "node_id": "kernel_sched_latency",
        "layer": "K",
        "metric": "node_schedstat_waiting_seconds_total",
        "has_inherent_defect": False,
        "node_description": "内核调度等待延迟"
    },
    {
        "node_id": "docker_mysql_cpu",
        "layer": "R",
        "metric": 'container_cpu_usage_seconds_total{id="/docker/3cd1fd44e7249fde61de089966171c372d566be97fd31553296ded55c437e8bf"}',
        "has_inherent_defect": False,
        "node_description": "MySQL容器CPU使用率"
    },
    {
        "node_id": "order_service_retries",
        "layer": "A",
        "metric": "app_retries_total",
        "has_inherent_defect": False,
        "node_description": "订单服务重试次数"
    }
]

def main():
    prom = PrometheusConnect(url=PROM_URL, disable_ssl=True)
    os.makedirs("bcpn_test_scenario/nodes", exist_ok=True)
    
    # 1. 生成每个节点的CSV文件
    for node in BCPN_NODES:
        print(f"采集节点：{node['node_id']}")
        # 修复点：step参数通过params字典传递，而非直接传入
        data = prom.get_metric_range_data(
            metric_name=node["metric"],
            start_time=START_TIME,
            end_time=END_TIME,
            params={"step": STEP}  # 核心修复：将step放入params参数
        )
        
        if not data:
            print(f"警告：未采集到节点 {node['node_id']} 的数据")
            continue
            
        # 转换为DataFrame
        df = pd.DataFrame(data[0]["values"], columns=["time", "value"])
        df["time"] = pd.to_datetime(df["time"], unit="s")
        df.to_csv(f"bcpn_test_scenario/nodes/{node['node_id']}.csv", index=False)
    
    # 2. 生成场景元数据文件
    scenario_metadata = {
        "scenario_info": {
            "scenario_id": "ssd_fault_with_retry_loop",
            "scenario_name": "SSD介质坏道叠加应用重试风暴",
            "fault_start_time": START_TIME.isoformat(),
            "fault_end_time": END_TIME.isoformat(),
            "affected_hosts": ["host01"]
        },
        "nodes": BCPN_NODES,
        "logs": []
    }
    
    with open("bcpn_test_scenario/scenario_metadata.json", "w", encoding="utf-8") as f:
        json.dump(scenario_metadata, f, ensure_ascii=False, indent=2)
    
    print("✅ BCPN格式数据生成完成，路径：./bcpn_test_scenario/")

if __name__ == "__main__":
    main()


In [ ]:
# 2. 运行数据采集
# 故障注入完成后，运行采集脚本
python3.10 generate_bcpn_data.py